In [ ]:
import hickle
import pandas as pd
import scipy.signal
# import shap
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import FormatStrFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
# from astropy.table import Table
from scipy import ndimage
from tqdm import tqdm
import corner
from PIL import Image
import io

import os

import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

from scipy.stats import chi2
from sklearn.datasets import make_s_curve
import matplotlib as mpl
import matplotlib.pyplot as plt
import tqdm

import ili
from ili.dataloaders import NumpyLoader
from ili.inference import InferenceRunner
from ili.validation.metrics import PosteriorCoverage, PlotSinglePosterior

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

from datasets import load_dataset, DatasetDict
from transformers import ViTFeatureExtractor, ViTForImageClassification, ViTModel, AdamW, AutoImageProcessor, AutoModel
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
from tqdm import tqdm

In [ ]:
GSWLC_data = load_dataset('juanpabloalfonzo/GSWLC_Data',split='train')
gswlc_data = pd.DataFrame(GSWLC_data)

gswlc_images = load_dataset('juanpabloalfonzo/GSWLC_Images',split='train')
gswlc_images = gswlc_images['images'].copy()

In [ ]:
# from transformers import AutoImageProcessor, AutoModel

# processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
# model = AutoModel.from_pretrained("google/vit-base-patch16-224").to(device)

# def infer(image):
#   inputs = processor(image, return_tensors="pt").to(device)
#   outputs = model(**inputs)
#   return outputs.pooler_output.cpu().detach().numpy().ravel()

In [ ]:
from transformers import AutoImageProcessor, AutoModel
import torch
import numpy as np

# CUDA Specific Determinism
# device = torch.device("cuda")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.cuda.manual_seed(42)
torch.manual_seed(42)

processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = AutoModel.from_pretrained("google/vit-base-patch16-224").to(device)
model.eval()

def infer(image):
    inputs = processor(image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.pooler_output.cpu().detach().numpy().ravel()


In [ ]:
new_x = []

ranges = [(9,12),(-3,3),(0,0.15),(0,13.7),(0,3),(6.5,9.0)]
labels = ['log M$_*$', 'log sSFR','redshift','t50','Av','Z']

for i in tqdm(range(len(gswlc_data))):
# for i in range(100):
    
    xobs_im = infer(Image.open(io.BytesIO(gswlc_images[i])))
    # posterior_samples_im = posterior_ensemble_im.sample((1000,), torch.Tensor(xobs_im).to(device), show_progress_bars=False)
    new_x.append(xobs_im)
    # posterior_samples_im = posterior_ensemble_im.sample((1000,), torch.Tensor(x_im[i]).to(device), show_progress_bars=False)

    # fig = corner.corner(posterior_samples_im.detach().numpy(), truths=theta_im[i], truth_color='k',color='firebrick',range=ranges,labels=labels,plot_datapoints=False, fill_contours=True,quantiles=(0.16, 0.5, 0.84),levels=[1 - np.exp(-(1/1)**2/2),1 - np.exp(-(2/1)**2/2)])
    # fig.set_size_inches(12,12)
    # plt.show()

gswlc_embeds = np.array(new_x)
gswlc_embeds.shape

In [ ]:
new_x = []
for i in tqdm(range(len(gswlc_data))):
# for i in range(100):
    monochromatic_image = np.array(Image.open(io.BytesIO(gswlc_images[i])))[:,:,2]
    # empty_image = np.zeros((256,256))
    monochromatic_image = np.stack((monochromatic_image,monochromatic_image,monochromatic_image),axis=2)
    xobs_im = infer(monochromatic_image)
    # posterior_samples_im = posterior_ensemble_im.sample((1000,), torch.Tensor(xobs_im).to(device), show_progress_bars=False)
    new_x.append(xobs_im)
    # posterior_samples_im = posterior_ensemble_im.sample((1000,), torch.Tensor(x_im[i]).to(device), show_progress_bars=False)

    # fig = corner.corner(posterior_samples_im.detach().numpy(), truths=theta_im[i], truth_color='k',color='firebrick',range=ranges,labels=labels,plot_datapoints=False, fill_contours=True,quantiles=(0.16, 0.5, 0.84),levels=[1 - np.exp(-(1/1)**2/2),1 - np.exp(-(2/1)**2/2)])
    # fig.set_size_inches(12,12)
    # plt.show()

gswlc_embeds_g_band = np.array(new_x)
gswlc_embeds_g_band.shape
hickle.dump(gswlc_embeds_g_band,'/home/juanpabloalfonzo/Documents/ViT Morph Fitting/ViT_embeds/gswlc_emebds_g_band_clean.hickle')

In [ ]:
gswlc_embeds_half_size = []
def resize_image_bytes(image_bytes, factor=0.5):
    """
    Resizes an image given as bytes to a specified factor of its original size.

    Args:
        image_bytes: The image data as a byte string.
        factor: The scaling factor for resizing (e.g., 0.5 for half size).

    Returns:
        The resized image data as a byte string, or None if an error occurs.
    """
    try:
        img_bytes = io.BytesIO(image_bytes)
        img = Image.open(img_bytes)

        width, height = img.size
        new_width = int(width * factor)
        new_height = int(height * factor)

        resized_img = img.resize((new_width, new_height))

        resized_img_bytes = io.BytesIO()
        resized_img.save(resized_img_bytes, format=img.format)
        resized_img_bytes.seek(0)

        return resized_img_bytes.getvalue()  # Return the byte string

    except Exception as e:
        print(f"Error resizing image: {e}")
        return None
    
half_size_images = []
for img in tqdm(gswlc_images):
    half_size_images.append(resize_image_bytes(img, factor=0.5))

for i in tqdm(range(len(gswlc_images))):
    gswlc_embeds_half_size.append(infer(Image.open(io.BytesIO(half_size_images[i]))))

gswlc_embeds_half_size = np.array(gswlc_embeds_half_size)
hickle.dump(gswlc_embeds_half_size,'/home/juanpabloalfonzo/Documents/ViT Morph Fitting/ViT_embeds/gswlc_emebds_half_size_clean.hickle' )

In [ ]:
len(gswlc_embeds)

In [ ]:
quarter_size_images = []
for img in tqdm(gswlc_images):
    quarter_size_images.append(resize_image_bytes(img, factor=0.25))

gswlc_embeds_quarter_size = []
for i in tqdm(range(len(gswlc_images))):
    gswlc_embeds_quarter_size.append(infer(Image.open(io.BytesIO(quarter_size_images[i]))))

gswlc_embeds_quarter_size = np.array(gswlc_embeds_quarter_size)
hickle.dump(gswlc_embeds_quarter_size,'/home/juanpabloalfonzo/Documents/ViT Morph Fitting/ViT_embeds/gswlc_emebds_quarter_size_clean.hickle' )

In [ ]:
eight_size_images = []
for img in tqdm(gswlc_images):
    eight_size_images.append(resize_image_bytes(img, factor=0.125))
gswlc_embeds_eight_size = []
for i in tqdm(range(len(gswlc_images))):
    gswlc_embeds_eight_size.append(infer(Image.open(io.BytesIO(eight_size_images[i]))))
gswlc_embeds_eight_size = np.array(gswlc_embeds_eight_size)
hickle.dump(gswlc_embeds_eight_size,'/home/juanpabloalfonzo/Documents/ViT Morph Fitting/ViT_embeds/gswlc_emebds_eight_size_clean.hickle' )

In [ ]:
gswlc_embeds_test = np.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/embed_test.npy')
# gswlc_embeds = np.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/ViT_embeds/gswlc_emebds_clean.npy')

plt.scatter(gswlc_embeds_test[0], gswlc_embeds[0])

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components = 200, random_state = 42)
x_im_transformed = pca.fit_transform(gswlc_embeds)

In [ ]:
plt.figure(figsize=(12,12))
plt.imshow(pca.get_covariance(), vmin=-0.01, vmax=0.01,cmap='Spectral')
plt.colorbar()

In [ ]:
temp_fuv = np.array(gswlc_data['FUV']).copy()
temp_fuv[temp_fuv > 25] = 25 + 0.3*np.random.normal(size=(np.sum(temp_fuv>25),))

temp_w2 = np.array(gswlc_data['W2']).copy()
temp_w2[temp_w2 > 23] = 23

temp_w3 = np.array(gswlc_data['W3']).copy()
temp_w3[temp_w3 > 23] = 23 + 0.3*np.random.normal(size=(np.sum(temp_w3>23),))

temp_w4 = np.array(gswlc_data['W4']).copy()
temp_w4[temp_w4 > 23] = 23 + 0.3*np.random.normal(size=(np.sum(temp_w4>23),))

eleven_bands_SED = np.array((temp_fuv, 
               gswlc_data['NUV'],
               gswlc_data['u'],
               gswlc_data['g'],
               gswlc_data['r'],
               gswlc_data['i'],
               gswlc_data['z'],
               gswlc_data['W1'],
               temp_w2,
               temp_w3,
               temp_w4,)).T

tempt50 = np.array(gswlc_data['t_50']).copy()
tempt50[(tempt50 > 13.7)] = 13.7

tempssfr = np.array(gswlc_data['logSSFR_50']).copy()
tempssfr[tempssfr < -3.5] = tempssfr[tempssfr < -3.5] + 0.3*np.random.normal(size=(np.sum(tempssfr < -3.5),))

tempmet = np.array(gswlc_data['logOH_50']).copy()
tempmet[tempmet < 7.1] = tempmet[tempmet < 7.1] + 0.1*np.random.normal(size=(np.sum(tempmet < 7.1),))

theta = np.vstack((gswlc_data['logM_50_x'], 
                   tempssfr,
                   gswlc_data['specz'],
                   tempt50,
                   gswlc_data['tauV_50'],
                   tempmet)).T
                #    manga_scalars['sersic_n'])).T
theta[np.isnan(theta)] = 0.

In [ ]:
five_bands_SED= np.array((
               gswlc_data['u'],
               gswlc_data['g'],
               gswlc_data['r'],
               gswlc_data['i'],
               gswlc_data['z'],
               )).T

In [ ]:
three_bands_SED = np.array((gswlc_data['g'],
                gswlc_data['r'],
                gswlc_data['i'],
                )).T

In [ ]:
def train_ili_post_ensemble(x, theta):
    
    loader = NumpyLoader(x=x, theta=theta)

    lower_bounds = torch.ones((theta.shape[1],))*(np.amin(theta,0)/1.2)
    upper_bounds = torch.ones((theta.shape[1],))*(np.amax(theta,0)*1.2)
    print(lower_bounds, upper_bounds)

    # define a prior
    prior = ili.utils.Uniform(low=lower_bounds, high=upper_bounds, device=device)

    # instantiate your neural networks to be used as an ensemble
    nets = [
        ili.utils.load_nde_sbi(engine='NPE', model='maf', hidden_features=50, num_transforms=5),
        ili.utils.load_nde_sbi(engine='NPE', model='mdn', hidden_features=50, num_components=6)
    ]

    # define training arguments
    train_args = {
        'training_batch_size': 32,
        'learning_rate': 1e-4
    }

    # initialize the trainer
    runner = InferenceRunner.load(
        backend='sbi',
        engine='NPE',
        prior=prior,
        nets=nets,
        device=device,
        # embedding_net=None,
        train_args=train_args,
        proposal=None,
        out_dir=None
    )

    # train the model
    posterior_ensemble, summaries = runner(loader=loader)
    
    return posterior_ensemble, summaries

In [ ]:
train_index = int(0.8*len(gswlc_data))
gswlc_embeds =  hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/ViT_embeds/gswlc_emebds_clean.hickle')

In [ ]:
posterior_ensemble_sed, summaries_sed = train_ili_post_ensemble(five_bands_SED[:train_index], theta[:train_index])

In [ ]:
posterior_ensemble_sed_11, summaries_sed_11 = train_ili_post_ensemble(eleven_bands_SED[:train_index], theta[:train_index])

In [ ]:
# posterior_ensemble_im, summaries_im, x_im, theta_im = hickle.load('ili/CPU_trained_ILI_GSWLC.hickle')

posterior_ensemble_im, summaries_im = train_ili_post_ensemble(gswlc_embeds[:train_index], theta[:train_index])

In [ ]:
posterior_ensemble_combined_5_bands, summaries_combined_5_bands = train_ili_post_ensemble(np.hstack((gswlc_embeds[:train_index],five_bands_SED[:train_index])), theta[:train_index])

In [ ]:
posterior_ensemble_combined_11_bands, summaries_combined_11_bands = train_ili_post_ensemble(np.hstack((gswlc_embeds[:train_index],eleven_bands_SED[:train_index])), theta[:train_index])

In [ ]:
gswlc_embeds_half_size = hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/ViT_embeds/gswlc_emebds_half_size_clean.hickle')
posterior_ensemble_im_half_size, summaries_im_half_size = train_ili_post_ensemble(gswlc_embeds_half_size[:train_index], theta[:train_index])
hickle.dump([posterior_ensemble_im_half_size, summaries_im_half_size],'/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_half_size.hickle')

In [ ]:
gswlc_embeds_quarter_size = hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/ViT_embeds/gswlc_emebds_quarter_size_clean.hickle')
posterior_ensemble_im_quarter_size, summaries_im_quarter_size = train_ili_post_ensemble(gswlc_embeds_quarter_size[:train_index], theta[:train_index])
hickle.dump([posterior_ensemble_im_quarter_size, summaries_im_quarter_size],'/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_quarter_size.hickle')

In [ ]:
gswlc_embeds_eight_size = hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/ViT_embeds/gswlc_emebds_eight_size_clean.hickle')
posterior_ensemble_im_eight_size, summaries_im_eight_size = train_ili_post_ensemble(gswlc_embeds_eight_size[:train_index], theta[:train_index])
hickle.dump([posterior_ensemble_im_eight_size, summaries_im_eight_size],'/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_eight_size.hickle')

In [ ]:
gswlc_embeds_g_band = hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/ViT_embeds/gswlc_emebds_g_band_clean.hickle')
posterior_ensemble_im_g_band, summaries_im_g_band = train_ili_post_ensemble(gswlc_embeds_g_band[:train_index], theta[:train_index])
hickle.dump([posterior_ensemble_im_g_band, summaries_im_g_band],'/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_g_band_images.hickle')

In [ ]:
hickle.dump([posterior_ensemble_sed, summaries_sed], '/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_5_bands_SED.hickle')

In [ ]:
hickle.dump([posterior_ensemble_sed_11, summaries_sed_11], '/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_11_bands_SED.hickle')

In [ ]:
hickle.dump([posterior_ensemble_im, summaries_im], '/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_images.hickle')

In [ ]:
hickle.dump([posterior_ensemble_combined_5_bands, summaries_combined_5_bands], '/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_combined_5_bands.hickle')

In [ ]:
hickle.dump([posterior_ensemble_combined_11_bands, summaries_combined_11_bands], '/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_combined_11_bands.hickle')

In [ ]:
hickle.dump(gswlc_embeds,'/home/juanpabloalfonzo/Documents/ViT Morph Fitting/ViT_embeds/gswlc_emebds_clean.hickle' )

# Checking Results

In [ ]:
posterior_ensemble_sed , summaries_sed = hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_11_bands_SED_train_set.hickle')
posterior_ensemble_im, summaries_im = hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_images_train_set.hickle')
posterior_ensemble_combined_5_bands, summaries_combined_5_bands = hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_combined_5_bands.hickle')
posterior_ensemble_combined_11_bands, summaries_combined_11_bands = hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_combined_11_bands.hickle')
gswlc_embeds =  hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/ViT_embeds/gswlc_emebds_clean.hickle')
train_index = int(0.8*len(gswlc_data))

In [ ]:
def check_degeneracy(samples, vb=False):

    covs = np.zeros((samples.shape[1],samples.shape[1]))
    pvals = np.ones((samples.shape[1],samples.shape[1]))
    for i in range(samples.shape[1]):
        for j in range(i):
            covs[i,j], pvals[i,j] = spearmanr(samples[:,i], samples[:,j])

    if vb==True:
        plt.imshow(np.abs(covs[1:,:-1]));
        plt.xticks(np.arange(5), labels[0:5])
        plt.yticks(np.arange(5), labels[1:])
        plt.colorbar(); plt.show()
        
        plt.imshow(np.abs((pvals<0.05)[1:,:-1]))
        plt.xticks(np.arange(5), labels[0:5])
        plt.yticks(np.arange(5), labels[1:])
        plt.colorbar(); plt.show()

    return covs[1:,:-1], pvals[1:,:-1]

In [ ]:
five_bands_SED_err= np.array((
                gswlc_data['eu'],
                gswlc_data['eg'],
                gswlc_data['er'],
                gswlc_data['ei'],
                gswlc_data['ez'],
                )).T

In [ ]:
eleven_bands_SED_err = np.array((gswlc_data['eFUV'],
               gswlc_data['eNUV'],
               gswlc_data['eu'],
               gswlc_data['eg'],
               gswlc_data['er'],
               gswlc_data['ei'],
               gswlc_data['ez'],
               gswlc_data['eW1'],
               gswlc_data['eW2'],
               gswlc_data['eW3'],
               gswlc_data['eW4'],)).T

In [ ]:
#plotting 5 band SED

plt.scatter(np.arange(0,5),five_bands_SED[train_index+3])
plt.xticks(np.arange(0,5),['u','g','r','i','z'])
plt.xlabel('Band')
plt.ylabel('Magnitude')

In [ ]:
#plotting 11 band SED
# plt.errorbar(np.arange(0,11),eleven_bands_SED[2], yerr = np.abs(eleven_bands_SED_err[2]), fmt='o', )
plt.scatter(np.arange(0,11),eleven_bands_SED[train_index+3])
plt.xticks(np.arange(0,11),['FUV','NUV','u','g','r','i','z','W1','W2','W3','W4'])
plt.xlabel('Band')
plt.ylabel('Magnitude')

In [ ]:
clrs = plt.cm.Spectral(np.linspace(0,1,10))
ranges = [(9,12),(-4.5,3),(0,0.16),(0,13.7),(0,3),(6.5,9.5)]
labels = ['log M$_*$', 'log sSFR','redshift','t50','Av','Z']

all_covs_sed, all_samples_sed = [], []
all_covs_im, all_samples_im = [], []

for gid in range(train_index, train_index+3):

    xobs_sed = eleven_bands_SED[gid,:]
    # xobs_sed = five_bands_SED[gid,:]
    # xobs_sed = three_bands_SED[gid,:]
    # xobs_im = infer(Image.open(io.BytesIO(gswlc_images[gid])))
    xobs_im = gswlc_embeds[gid,:]
    x_truth = theta[gid,:]

    posterior_samples_sed = posterior_ensemble_sed.sample((1000,), torch.Tensor(xobs_sed).to(device), show_progress_bars = False)
    posterior_samples_sed = posterior_samples_sed.cpu().numpy()
    # covs_sed, _ = check_degeneracy(posterior_samples_sed, vb=True)

    posterior_samples_im = posterior_ensemble_im.sample((1000,), torch.Tensor(xobs_im).to(device), show_progress_bars = False)
    posterior_samples_im = posterior_samples_im.cpu().numpy()
    # covs_im, _ = check_degeneracy(posterior_samples_im, vb=True)

    # all_covs_sed.append(covs_sed)
    # all_samples_sed.append(posterior_samples_sed)
    # all_covs_im.append(covs_im)
    # all_samples_im.append(posterior_samples_im)

    plt.imshow(Image.open(io.BytesIO(gswlc_images[gid])))
    plt.axis('off')

    fig = corner.corner(posterior_samples_sed, truths=x_truth,color='dodgerblue',range=ranges,labels=labels, plot_datapoints=False, fill_contours=True,quantiles=(0.16, 0.5, 0.84),levels=[1 - np.exp(-(1/1)**2/2),1 - np.exp(-(2/1)**2/2)])
    fig = corner.corner(posterior_samples_im, truths=x_truth, truth_color='k',color='firebrick',range=ranges,labels=labels,fig=fig, plot_datapoints=False, fill_contours=True,quantiles=(0.16, 0.5, 0.84),levels=[1 - np.exp(-(1/1)**2/2),1 - np.exp(-(2/1)**2/2)])
    fig.set_size_inches(12,12)
    plt.show()

# Checking Degeneracies in Predictions

In [ ]:

clrs = plt.cm.Spectral(np.linspace(0,1,10))
ranges = [(9,12),(-15,-8),(0,0.15),(0,13.7),(0,3),(-1.5,0.25)]
labels = ['log M$_*$', 'log SFR','redshift','t50','Av','Z']

all_covs_sed, all_samples_sed = [], []
all_covs_im, all_samples_im = [], []

for gid in tqdm(range(train_index, len(gswlc_data))):
    xobs_sed = eleven_bands_SED[gid,:]
    # xobs_sed = five_bands_SED[gid,:]
    # xobs_im = infer(Image.open(io.BytesIO(gswlc_images[gid])))
    xobs_im = gswlc_embeds[gid,:]
    x_truth = theta[gid,:]

    posterior_samples_sed = posterior_ensemble_sed.sample((1000,), torch.Tensor(xobs_sed).to(device), show_progress_bars = False)
    posterior_samples_sed = posterior_samples_sed.cpu().numpy()
    covs_sed, _ = check_degeneracy(posterior_samples_sed)

    posterior_samples_im = posterior_ensemble_im.sample((1000,), torch.Tensor(xobs_im).to(device), show_progress_bars = False)
    posterior_samples_im = posterior_samples_im.cpu().numpy()
    covs_im, _ = check_degeneracy(posterior_samples_im)

    all_covs_sed.append(covs_sed)
    all_samples_sed.append(posterior_samples_sed)
    all_covs_im.append(covs_im)
    all_samples_im.append(posterior_samples_im)

In [ ]:
#combined input models
clrs = plt.cm.Spectral(np.linspace(0,1,10))
ranges = [(9,12),(-15,-8),(0,0.15),(0,13.7),(0,3),(-1.5,0.25)]
labels = ['log M$_*$', 'log SFR','redshift','t50','Av','Z']

all_covs_sed, all_samples_sed = [], []


for gid in tqdm(range(train_index, len(gswlc_data))):
    xobs_sed = eleven_bands_SED[gid,:]
    # xobs_sed = five_bands_SED[gid,:]
    # xobs_im = infer(Image.open(io.BytesIO(gswlc_images[gid])))
    xobs_im = gswlc_embeds[gid,:]
    x_truth = theta[gid,:]

    posterior_samples_sed = posterior_ensemble_combined_11_bands.sample((1000,), torch.Tensor(np.hstack((xobs_im,xobs_sed))).to(device), show_progress_bars = False)
    posterior_samples_sed = posterior_samples_sed.cpu().numpy()
    covs_sed, _ = check_degeneracy(posterior_samples_sed)


    all_covs_sed.append(covs_sed)
    all_samples_sed.append(posterior_samples_sed)

In [ ]:
hickle.dump([all_covs_sed, all_samples_sed], '/home/juanpabloalfonzo/Documents/ViT Morph Fitting/samples_and_covs/combined_11_bands_samples_and_covs.hickle')
# hickle.dump([all_covs_im, all_samples_im], '/home/juanpabloalfonzo/Documents/ViT Morph Fitting/samples_and_covs/images_samples_and_covs.hickle')

In [ ]:
# hickle.dump([all_covs_sed, all_samples_sed], '/home/juanpabloalfonzo/Documents/ViT Morph Fitting/samples_and_covs/5_bands_SED_samples_and_covs.hickle')

In [ ]:
all_covs_sed, all_samples_sed = hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/samples_and_covs/11_bands_SED_samples_and_covs.hickle')
all_covs_im, all_samples_im = hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/samples_and_covs/images_samples_and_covs.hickle')

cov_array_sed = np.array(all_covs_sed)
cov_array_im = np.array(all_covs_im)
cov_array_sed.shape, cov_array_im.shape

all_covs_sed_5_bands, all_samples_sed_5_bands = hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/samples_and_covs/5_bands_SED_samples_and_covs.hickle')

cov_array_sed_5_bands = np.array(all_covs_sed_5_bands)
cov_array_sed_5_bands.shape

train_index = int(0.8*len(gswlc_data))
sample_size = len(gswlc_data) - train_index

In [ ]:
import seaborn as sns
labels = ['log M$_*$', 'log SFR','z','t$_{50}$','Dust','Metallicity']

target_sed = cov_array_sed_5_bands

plt.figure(figsize=(9*2,7))
plt.subplot(1,2,1)
# plt.imshow(np.nanmedian(np.abs(target_sed),0), cmap='coolwarm',vmin=0., vmax=1.)
# plt.xticks(np.arange(5), labels[0:5], fontsize=18)
# plt.yticks(np.arange(5), labels[1:], fontsize=18)
sns.heatmap(np.nanmedian(np.abs(target_sed),0), cmap='coolwarm',vmin=0., vmax=1., annot=True , cbar=True)
plt.xticks(np.arange(5)+0.5, labels[0:5], fontsize=16)
plt.yticks(np.arange(5)+0.5, labels[1:], fontsize=16)
plt.title('median covariances [SED]',fontsize=18)
# plt.colorbar(); 
plt.subplot(1,2,2)
# plt.imshow(np.nanmedian(np.abs(cov_array_im),0), cmap='coolwarm',vmin=0., vmax=1.)
# plt.xticks(np.arange(5), labels[0:5], fontsize=18)
# plt.yticks(np.arange(5), labels[1:], fontsize=18)
sns.heatmap(np.nanmedian(np.abs(cov_array_im),0), cmap='coolwarm',vmin=0., vmax=1., annot=True , cbar=True)
plt.xticks(np.arange(5)+0.5, labels[0:5], fontsize=16)
plt.yticks(np.arange(5)+0.5, labels[1:], fontsize=16)
plt.title('median covariances [Image]',fontsize=18)
# plt.colorbar(); 
plt.tight_layout()
plt.show()

plt.figure(figsize=(9*2,7))
plt.subplot(1,2,1)
# plt.imshow(np.nanstd(np.abs(target_sed),0), cmap='coolwarm')
# plt.xticks(np.arange(5), labels[0:5], fontsize=18)
# plt.yticks(np.arange(5), labels[1:], fontsize=18)
sns.heatmap(np.nanstd(np.abs(target_sed),0), cmap='coolwarm',vmin=0., vmax=1., annot=True , cbar=True)
plt.xticks(np.arange(5)+0.5, labels[0:5], fontsize=16)
plt.yticks(np.arange(5)+0.5, labels[1:], fontsize=16)
plt.title('std covariances [SED]',fontsize=18)
# plt.colorbar(); 
plt.subplot(1,2,2)
# plt.imshow(np.nanstd(np.abs(cov_array_im),0), cmap='coolwarm')
# plt.xticks(np.arange(5), labels[0:5], fontsize=18)
# plt.yticks(np.arange(5), labels[1:], fontsize=18)
sns.heatmap(np.nanstd(np.abs(cov_array_im),0), cmap='coolwarm',vmin=0., vmax=1., annot=True , cbar=True)
plt.xticks(np.arange(5)+0.5, labels[0:5], fontsize=16)
plt.yticks(np.arange(5)+0.5, labels[1:], fontsize=16)
plt.title('std covariances [Image]',fontsize=18)
# plt.colorbar(); 
plt.tight_layout()
plt.show()

In [ ]:
print(len(all_samples_sed), sample_size)

In [ ]:
sample_size = len(gswlc_data) - train_index

target_sed = all_samples_sed_5_bands

for i in range(6):
    for j in range(i):
        print('covs between',labels[i],' and',labels[j])
        
        alli_im, allj_im, alli_sed, allj_sed = [], [], [], []
        for gid in range(sample_size):
            alli_im.append(all_samples_im[gid][:,i] - theta[gid+train_index, i])
            allj_im.append(all_samples_im[gid][:,j] - theta[gid+train_index, j])
            alli_sed.append(target_sed[gid][:,i] - theta[gid+train_index, i])
            allj_sed.append(target_sed[gid][:,j] - theta[gid+train_index, j])
        
        alli_im, allj_im, alli_sed, allj_sed = np.hstack(alli_im), np.hstack(allj_im), np.hstack(alli_sed), np.hstack(allj_sed)
        rval = 4
        
        
        fig = corner.corner(np.vstack([alli_sed, allj_sed]).T, 
                      truths=[0,0], range=((-rval,rval),(-rval,rval)),
                      color='dodgerblue',
                      labels=('$\Delta$ '+labels[i],'$\Delta$ '+labels[j]), 
                      plot_datapoints=False, 
                      fill_contours=True,
                      # quantiles=(0.16, 0.5, 0.84),
                      levels=[1 - np.exp(-(1/1)**2/2),1 - np.exp(-(2/1)**2/2)])
        
        fig = corner.corner(np.vstack([alli_im, allj_im]).T, 
                      truths=[0,0], fig=fig, range=((-rval,rval),(-rval,rval)),
                      color='firebrick',
                      labels=('$\Delta$ '+labels[i],'$\Delta$ '+labels[j]), 
                      plot_datapoints=False, 
                      fill_contours=True,
                      # quantiles=(0.16, 0.5, 0.84),
                      levels=[1 - np.exp(-(1/1)**2/2),1 - np.exp(-(2/1)**2/2)])
        
        plt.show()

In [ ]:
# for i in range(6):
#     for j in range(i):
#         print('covs between',labels[i],' and',labels[j])
        
#         alli_im, allj_im, alli_sed, allj_sed, alli_im15, allj_im15, alli_im50, allj_im50 = [], [], [], [], [], [], [], []
#         for gid in range(sample_size):
#             alli_im.append(all_samples_im[gid][:,i] - theta[gid, i])
#             allj_im.append(all_samples_im[gid][:,j] - theta[gid, j])
#             alli_im15.append(all_samples_im_pca15[gid][:,i] - theta[gid, i])
#             allj_im15.append(all_samples_im_pca15[gid][:,j] - theta[gid, j])
#             alli_im50.append(all_samples_im_pca50[gid][:,i] - theta[gid, i])
#             allj_im50.append(all_samples_im_pca50[gid][:,j] - theta[gid, j])
#             alli_sed.append(all_samples_sed[gid][:,i] - theta[gid, i])
#             allj_sed.append(all_samples_sed[gid][:,j] - theta[gid, j])
        
#         alli_im, allj_im, alli_sed, allj_sed = np.hstack(alli_im), np.hstack(allj_im), np.hstack(alli_sed), np.hstack(allj_sed)
#         alli_im15, allj_im15, alli_im50, allj_im50 = np.hstack(alli_im15), np.hstack(allj_im15), np.hstack(alli_im50), np.hstack(allj_im50)
#         rval = 4
        
        
#         fig = corner.corner(np.vstack([alli_sed, allj_sed]).T, 
#                       truths=[0,0], range=((-rval,rval),(-rval,rval)),
#                       color='dodgerblue',
#                       labels=('$\Delta$ '+labels[i],'$\Delta$ '+labels[j]), 
#                       plot_datapoints=False, 
#                       fill_contours=True,
#                       # quantiles=(0.16, 0.5, 0.84),
#                       levels=[1 - np.exp(-(1/1)**2/2),1 - np.exp(-(2/1)**2/2)])
        
#         fig = corner.corner(np.vstack([alli_im, allj_im]).T, 
#                       truths=[0,0], fig=fig, range=((-rval,rval),(-rval,rval)),
#                       color='firebrick',
#                       labels=('$\Delta$ '+labels[i],'$\Delta$ '+labels[j]), 
#                       plot_datapoints=False, 
#                       fill_contours=True,
#                       # quantiles=(0.16, 0.5, 0.84),
#                       levels=[1 - np.exp(-(1/1)**2/2),1 - np.exp(-(2/1)**2/2)])

#         fig = corner.corner(np.vstack([alli_im15, allj_im15]).T, 
#                       truths=[0,0], fig=fig, range=((-rval,rval),(-rval,rval)),
#                       color='forestgreen',
#                       labels=('$\Delta$ '+labels[i],'$\Delta$ '+labels[j]), 
#                       plot_datapoints=False, 
#                       fill_contours=True,
#                       # quantiles=(0.16, 0.5, 0.84),
#                       levels=[1 - np.exp(-(1/1)**2/2),1 - np.exp(-(2/1)**2/2)])

#         fig = corner.corner(np.vstack([alli_im50, allj_im50]).T, 
#                       truths=[0,0], fig=fig, range=((-rval,rval),(-rval,rval)),
#                       color='deeppink',
#                       labels=('$\Delta$ '+labels[i],'$\Delta$ '+labels[j]), 
#                       plot_datapoints=False, 
#                       fill_contours=True,
#                       # quantiles=(0.16, 0.5, 0.84),
#                       levels=[1 - np.exp(-(1/1)**2/2),1 - np.exp(-(2/1)**2/2)])
        
#         plt.show()

# are there any examples of better constraints with images?

In [ ]:
nqty = 6
allsederr, allimerr, allsedstd, allimstd = [], [], [], []

allimerr_raw = []
allsederr_raw = []
allimpred, allsedpred = [], []

target_sed = all_samples_sed_5_bands

for gid in range(sample_size):
    
    sed_err, im_err, sed_std, im_std, sed_err_raw, im_err_raw, im_pred, sed_pred = np.zeros((nqty,)), np.zeros((nqty,)), np.zeros((nqty,)), np.zeros((nqty,)), np.zeros((nqty,)), np.zeros((nqty,)), np.zeros((nqty,)), np.zeros((nqty,))
    
    for i in range(nqty):
            
        q1_im = (all_samples_im[gid][:,i] - theta[train_index+gid, i])
        q1_sed = (target_sed[gid][:,i] - theta[train_index+gid, i])
        sed_err[i] = np.nanmean(np.sqrt(q1_sed**2))
        sed_std[i] = np.nanstd(q1_sed)
        im_err[i] = np.nanmean(np.sqrt(q1_im**2))
        im_std[i] = np.nanstd(q1_im)
        im_err_raw[i] = np.nanmean(q1_im)
        sed_err_raw[i] = np.nanmean(q1_sed)
        im_pred[i] = np.nanmean(all_samples_im[gid][:,i])
        sed_pred[i] = np.nanmean(target_sed[gid][:,i])
    allsederr.append(sed_err)
    allsedstd.append(sed_std)
    allimerr.append(im_err)
    allimstd.append(im_std)
    allimerr_raw.append(im_err_raw)
    allsederr_raw.append(sed_err_raw)
    allimpred.append(im_pred)
    allsedpred.append(sed_pred)
    
allsederr = np.array(allsederr)
allsedstd = np.array(allsedstd)
allimerr = np.array(allimerr)
allimstd = np.array(allimstd)
allsederr_raw = np.array(allsederr_raw)
allimerr_raw = np.array(allimerr_raw)
allsedpred = np.array(allsedpred)
allimpred = np.array(allimpred)

#Rename all the arrays above to 5 bands in their name
allsederr_5_bands = allsederr
allsedstd_5_bands = allsedstd
allimerr_5_bands = allimerr
allimstd_5_bands = allimstd
allsederr_raw_5_bands = allsederr_raw
allimerr_raw_5_bands = allimerr_raw
allsedpred_5_bands = allsedpred
allimpred_5_bands = allimpred


In [ ]:
# mvals = theta[0:1000,0]
# sfrvals = theta[0:1000,1] + theta[0:1000,0] - 9

# mvals = theta[0:10000,0]
# sfrvals = theta[0:10000,1] + theta[0:10000,0] - 9

labels = ['log M$_*$', 'log SFR','redshift','t50','Av','Z']

mvals = theta[train_index:,0]
sfrvals = theta[train_index:,1] + theta[train_index:,0] - 9


for qty in range(6):
    c_lower_bound = np.percentile(allimerr[:,qty],2)
    c_upper_bound = np.percentile(allimerr[:,qty],98)
    plt.figure(figsize=(14,5))
    plt.subplot(1,2,1)
    plt.hexbin(mvals,sfrvals,gridsize=20,mincnt=2, C = allsederr_5_bands[:,qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    plt.xlabel('log Stellar Mass',fontsize=18); plt.ylabel('log SFR',fontsize=18)
    clbr = plt.colorbar()
    templim = plt.gci().get_clim()
    clbr.set_label('$\Delta$'+labels[qty]+'$_{,SED}$',fontsize=18)
    plt.subplot(1,2,2)
    plt.hexbin(mvals,sfrvals,gridsize=20,mincnt=2, C = allimerr[:,qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    plt.xlabel('log Stellar Mass',fontsize=18); plt.ylabel('log SFR',fontsize=18)
    clbr = plt.colorbar()
    clbr.set_label('$\Delta$'+labels[qty]+'$_{,Image}$',fontsize=18)
    plt.tight_layout()
    plt.show()

In [ ]:
# mvals = theta[0:1000,0]
# sfrvals = theta[0:1000,1] + theta[0:1000,0] - 9

# mvals = theta[0:10000,0]
# sfrvals = theta[0:10000,1] + theta[0:10000,0] - 9


mvals = theta[train_index:,0]
sfrvals = theta[train_index:,1] + theta[train_index:,0] - 9

for qty in range(6):
    c_lower_bound = np.percentile(allimstd[:,qty],2)
    c_upper_bound = np.percentile(allimstd[:,qty],98)
    plt.figure(figsize=(14,5))
    plt.subplot(1,2,1)
    plt.hexbin(mvals,sfrvals,gridsize=20,mincnt=2, C = allsedstd[:,qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    plt.xlabel('log Stellar Mass',fontsize=18); plt.ylabel('log SFR',fontsize=18)
    clbr = plt.colorbar()
    templim = plt.gci().get_clim()
    clbr.set_label('$\sigma$ '+labels[qty]+'$_{,SED}$',fontsize=18)
    plt.subplot(1,2,2)
    plt.hexbin(mvals,sfrvals,gridsize=20,mincnt=2, C = allimstd[:,qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    plt.xlabel('log Stellar Mass',fontsize=18); plt.ylabel('log SFR',fontsize=18)
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$ '+labels[qty]+'$_{,Image}$',fontsize=18)
    plt.tight_layout()
    plt.show()

# 11 Band Comparison

In [ ]:
nqty = 6
allsederr, allimerr, allsedstd, allimstd = [], [], [], []

allimerr_raw = []
allsederr_raw = []
allimpred, allsedpred = [], []

target_sed = all_samples_sed

for gid in range(sample_size):
    
    sed_err, im_err, sed_std, im_std, sed_err_raw, im_err_raw, im_pred, sed_pred = np.zeros((nqty,)), np.zeros((nqty,)), np.zeros((nqty,)), np.zeros((nqty,)), np.zeros((nqty,)), np.zeros((nqty,)), np.zeros((nqty,)), np.zeros((nqty,))
    
    for i in range(nqty):
            
        q1_im = (all_samples_im[gid][:,i] - theta[train_index+gid, i])
        q1_sed = (target_sed[gid][:,i] - theta[train_index+gid, i])
        sed_err[i] = np.nanmean(np.sqrt(q1_sed**2))
        sed_std[i] = np.nanstd(q1_sed)
        im_err[i] = np.nanmean(np.sqrt(q1_im**2))
        im_std[i] = np.nanstd(q1_im)
        im_err_raw[i] = np.nanmean(q1_im)
        sed_err_raw[i] = np.nanmean(q1_sed)
        im_pred[i] = np.nanmean(all_samples_im[gid][:,i])
        sed_pred[i] = np.nanmean(target_sed[gid][:,i])
    allsederr.append(sed_err)
    allsedstd.append(sed_std)
    allimerr.append(im_err)
    allimstd.append(im_std)
    allimerr_raw.append(im_err_raw)
    allsederr_raw.append(sed_err_raw)
    allimpred.append(im_pred)
    allsedpred.append(sed_pred)
    
allsederr = np.array(allsederr)
allsedstd = np.array(allsedstd)
allimerr = np.array(allimerr)
allimstd = np.array(allimstd)
allsederr_raw = np.array(allsederr_raw)
allimerr_raw = np.array(allimerr_raw)
allsedpred = np.array(allsedpred)
allimpred = np.array(allimpred)

#Rename all the arrays above to 11 bands in their name
allsederr_11_bands = allsederr
allsedstd_11_bands = allsedstd
allimerr_11_bands = allimerr
allimstd_11_bands = allimstd
allsederr_raw_11_bands = allsederr_raw
allimerr_raw_11_bands = allimerr_raw
allsedpred_11_bands = allsedpred
allimpred_11_bands = allimpred


In [ ]:
# mvals = theta[0:1000,0]
# sfrvals = theta[0:1000,1] + theta[0:1000,0] - 9

# mvals = theta[0:10000,0]
# sfrvals = theta[0:10000,1] + theta[0:10000,0] - 9
labels = ['log M$_*$', 'log SFR','redshift','t50','Av','Z']
mvals = theta[train_index:,0]
sfrvals = theta[train_index:,1] + theta[train_index:,0] - 9

dustvals = theta[0:sample_size,4]

for qty in range(6):
    c_lower_bound = np.percentile(allimerr[:,qty],2)
    c_upper_bound = np.percentile(allimerr[:,qty],98)
    plt.figure(figsize=(14,5))
    plt.subplot(1,2,1)
    plt.hexbin(mvals,sfrvals,gridsize=20,mincnt=2, C = allsederr[:,qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    plt.xlabel('log Stellar Mass',fontsize=18); plt.ylabel('log SFR',fontsize=18)
    clbr = plt.colorbar()
    templim = plt.gci().get_clim()
    clbr.set_label('$\Delta$'+labels[qty]+'$_{,SED}$',fontsize=18)
    plt.subplot(1,2,2)
    plt.hexbin(mvals,sfrvals,gridsize=20,mincnt=2, C = allimerr[:,qty], vmin=c_lower_bound, vmax=c_upper_bound,cmap='viridis')
    plt.xlabel('log Stellar Mass',fontsize=18); plt.ylabel('log SFR',fontsize=18)
    clbr = plt.colorbar()
    clbr.set_label('$\Delta$'+labels[qty]+'$_{,Image}$',fontsize=18)
    plt.tight_layout()
    plt.show()

In [ ]:
# mvals = theta[0:1000,0]
# sfrvals = theta[0:1000,1] + theta[0:1000,0] - 9

# mvals = theta[0:10000,0]
# sfrvals = theta[0:10000,1] + theta[0:10000,0] - 9

mvals = theta[train_index:,0]
sfrvals = theta[train_index:,1] + theta[train_index:,0] - 9

for qty in range(6):
    c_lower_bound = np.percentile(allimstd[:,qty],2)
    c_upper_bound = np.percentile(allimstd[:,qty],98)
    plt.figure(figsize=(14,5))
    plt.subplot(1,2,1)
    plt.hexbin(mvals,sfrvals,gridsize=20,mincnt=2, C = allsedstd[:,qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    plt.xlabel('log Stellar Mass',fontsize=18); plt.ylabel('log SFR',fontsize=18)
    clbr = plt.colorbar()
    templim = plt.gci().get_clim()
    clbr.set_label('$\sigma$ '+labels[qty]+'$_{,SED}$',fontsize=18)
    plt.subplot(1,2,2)
    plt.hexbin(mvals,sfrvals,gridsize=20,mincnt=2, C = allimstd[:,qty], vmin=c_lower_bound, vmax=c_upper_bound,cmap='viridis')
    plt.xlabel('log Stellar Mass',fontsize=18); plt.ylabel('log SFR',fontsize=18)
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$ '+labels[qty]+'$_{,Image}$',fontsize=18)
    plt.tight_layout()
    plt.show()

# Science Cases

SFMS Scatter Caused by Morphology

In [ ]:
lower_bound_mass = 10
upper_mound_mass = 11

main_sequence_sfr_sed = (0.84-0.026*12.5)*allsedpred_5_bands[:,0] - (6.51-0.11*12.5)
main_sequence_sfr_im = (0.84-0.026*12.5)*allimpred[:,0] - (6.51-0.11*12.5)
main_sequence_sfr_true= (0.84-0.026*12.5)*theta[train_index:,0] - (6.51-0.11*12.5)

sfr_vals_im = allimpred[:,1]+allimpred[:,0]-9
sfr_vals_sed = allsedpred_5_bands[:,1]+allsedpred_5_bands[:,0]-9
sfr_vals_true = theta[train_index:,1]+theta[train_index:,0]-9

mvals_im= allimpred[:,0]
mvals_sed = allsedpred_5_bands[:,0]
mvals_true = theta[train_index:,0]

# mask = np.where((mvals>lower_bound_mass)&(mvals<upper_mound_mass)&(sfrvals>=main_sequence_sfr) & (sersic_n>=4))[0]
# # mask = np.arange(len(mvals))
# print(len(mask))
# # plt.figure()
# # plt.scatter(mvals[mask],sfrvals[mask], cmap='viridis')
# # plt.scatter(mvals, sfrvals, color='red', alpha=0.005)


# plt.figure()
# plt.hist(allimstd[mask,1], label='Image', alpha=0.5)
# plt.hist(allsedstd_5_bands[mask,1],alpha=0.5,label='SED 5 bands')
# plt.hist(allsedstd_11_bands[mask,1],alpha=0.5, label='SED 11 bands')
# plt.legend()
# print(np.nanmean(np.array(allimerr_raw)[mask,1]))
# print(np.nanmean(np.array(allsederr_raw_5_bands)[mask,1]))
# print(np.nanmean(np.array(allsederr_raw_11_bands)[mask,1]))


mask_sed = np.where((sfr_vals_sed>=main_sequence_sfr_sed-0.5)&((sfr_vals_sed<=main_sequence_sfr_sed+0.5)))[0]
plt.figure()
plt.scatter(mvals_sed[mask_sed],sfr_vals_sed[mask_sed])
plt.plot(allsedpred_5_bands[:,0],main_sequence_sfr_sed,'--',color='black')

mask_im = np.where((sfr_vals_im>=main_sequence_sfr_im-0.5)&((sfr_vals_im<=main_sequence_sfr_im+0.5)))[0]
plt.figure()
plt.scatter(mvals_im[mask_im],sfr_vals_im[mask_im])
plt.plot(mvals_im,main_sequence_sfr_im,'--',color='black')

mask_true = np.where((sfr_vals_true>=main_sequence_sfr_true-0.5)&((sfr_vals_true<=main_sequence_sfr_true+0.5)))[0]
plt.figure()
plt.scatter(mvals_true[mask_true],sfr_vals_true[mask_true])
plt.plot(mvals_true,main_sequence_sfr_true,'--',color='black')

#Calculate scatter from sfms for 11 band SED and images
scatter_5_band_sed = sfr_vals_sed - main_sequence_sfr_sed
scatter_im = sfr_vals_im - main_sequence_sfr_im
scatter_true = sfr_vals_true - main_sequence_sfr_true


plt.figure()
plt.hist(scatter_5_band_sed, bins=50, label='SED 5 bands', alpha=0.3)
plt.hist(scatter_im, bins=50, label='Image', alpha=0.3)
plt.hist(scatter_true, bins=50, label='True', alpha=0.3)
plt.legend()
plt.xlabel('Scatter from SFMS', fontsize=18)


In [ ]:
len(scatter_true), len(scatter_im)

In [ ]:
param = 5
np.nanmean(allimerr_raw[:,param]), np.nanmean(allsederr_raw_5_bands[:,param]), np.nanmean(allsederr_raw_11_bands[:,param]), np.nanstd(allimerr_raw[:,param]), np.nanstd(allsederr_raw_5_bands[:,param]), np.nanstd(allsederr_raw_11_bands[:,param])

In [ ]:
plt.hist(allimerr_raw[:,1], label='Image', alpha=0.5)
plt.hist(allsederr_raw_5_bands[:,1],alpha=0.5,label='SED 5 bands')
plt.hist(allsederr_raw_11_bands[:,1],alpha=0.5, label='SED 11 bands')
plt.legend()

In [ ]:
mass_bins = np.arange(8.5,11,0.4)

plt.figure()
plt.title('Image Derived Values',fontsize=18)
plt.hexbin(mvals_im[mask_im],scatter_im[mask_im], C = pbulge[mask_im], gridsize=30, cmap='viridis', vmin=0.1, vmax=1)
plt.vlines(mass_bins, -0.6, 0.6, color='black', alpha=0.5)
# plt.hlines(0, 8.5, 11.5, color='black', alpha=0.8, linestyle='--')
plt.xlabel('log M*',fontsize=18)
plt.ylabel('$\Delta$log(SFR)',fontsize=18)
plt.colorbar(label='Prominent Bulge')

plt.figure()
plt.title('Photometery Derived Values',fontsize=18)
plt.hexbin(mvals_true[mask_true],scatter_true[mask_true], C = pbulge[mask_true], gridsize=30, cmap='viridis', vmin=0, vmax=1)
plt.vlines(mass_bins, -0.6, 0.6, color='black', alpha=0.8)
plt.xlabel('log M*',fontsize=18)
plt.ylabel('$\Delta$log(SFR)',fontsize=18)
plt.colorbar(label='Prominent Bulge')


plt.figure()
plt.title('SED Derived Values',fontsize=18)
plt.hexbin(mvals_sed[mask_sed],scatter_5_band_sed[mask_sed], C = pbulge[mask_sed], gridsize=30, cmap='viridis', vmin=0, vmax=1)
plt.vlines(mass_bins, -0.6, 0.6, color='black', alpha=0.8)
plt.xlabel('log M*',fontsize=18)
plt.ylabel('$\Delta$log(SFR)',fontsize=18)
plt.colorbar(label='Prominent Bulge')

In [ ]:
mass_bins = np.arange(8.5,11,0.4)

plt.figure()
plt.title('Image Derived Values',fontsize=18)
plt.hexbin(mvals_im[mask_im],scatter_im[mask_im], C = ttype[mask_im], gridsize=20, cmap='viridis' , vmin=-2,vmax=6)
plt.vlines(mass_bins, -0.6, 0.6, color='black', alpha=0.5)
# plt.hlines(0, 8.5, 11.5, color='black', alpha=0.8, linestyle='--')
plt.xlabel('log M*',fontsize=18)
plt.ylabel('$\Delta$log(SFR)',fontsize=18)
plt.colorbar(label='T Type')

plt.figure()
plt.title('Photometery Derived Values',fontsize=18)
plt.hexbin(mvals_true[mask_true],scatter_true[mask_true], C = ttype[mask_true], gridsize=20, cmap='viridis', vmin=-2, vmax=6)
plt.vlines(mass_bins, -0.6, 0.6, color='black', alpha=0.8)
plt.xlabel('log M*',fontsize=18)
plt.ylabel('$\Delta$log(SFR)',fontsize=18)
plt.colorbar(label='T Type')


plt.figure()
plt.title('SED Derived Values',fontsize=18)
plt.hexbin(mvals_sed[mask_sed],scatter_5_band_sed[mask_sed], C = ttype[mask_sed], gridsize=20, cmap='viridis',vmin=-2, vmax=6)
plt.vlines(mass_bins, -0.6, 0.6, color='black', alpha=0.8)
plt.xlabel('log M*',fontsize=18)
plt.ylabel('$\Delta$log(SFR)',fontsize=18)
plt.colorbar(label='T Type')

In [ ]:
mass_bins = np.arange(8.5,11,0.4)

plt.figure()
plt.title('Image Derived Values',fontsize=18)
plt.hexbin(mvals_im,scatter_im, C = ttype, gridsize=30, cmap='viridis')
# plt.vlines(mass_bins, -0.6, 0.6, color='black', alpha=0.5)
# plt.hlines(0, 8.5, 11.5, color='black', alpha=0.8, linestyle='--')
plt.xlabel('log M*',fontsize=18)
plt.ylabel('$\Delta$log(SFR)',fontsize=18)
plt.colorbar(label='Prominent Bulge')

plt.figure()
plt.title('Photometery Derived Values',fontsize=18)
plt.hexbin(mvals_true,scatter_true, C = ttype, gridsize=30, cmap='viridis')
# plt.vlines(mass_bins, -0.6, 0.6, color='black', alpha=0.8)
plt.xlabel('log M*',fontsize=18)
plt.ylabel('$\Delta$log(SFR)',fontsize=18)
plt.colorbar(label='Prominent Bulge')


plt.figure()
plt.title('SED Derived Values',fontsize=18)
plt.hexbin(mvals_sed,scatter_5_band_sed, C = ttype, gridsize=30, cmap='viridis')
# plt.vlines(mass_bins, -0.6, 0.6, color='black', alpha=0.8)
plt.xlabel('log M*',fontsize=18)
plt.ylabel('$\Delta$log(SFR)',fontsize=18)
plt.colorbar(label='Prominent Bulge')

Dusty galaxies

In [ ]:
#Very Dusty Galaxies
better_img = np.where((np.array(theta)[train_index:][:,4]>3.5)  & (allsederr[:,1]-allimerr[:,1]>0)  & (allsederr[:,3]-allimerr[:,3]>0) & (allsederr[:,4]-allimerr[:,4]>0) & (allsederr[:,5]-allimerr[:,5]>0))[0]
better_sed = np.where((np.array(theta)[train_index:][:,4]>3.5) & (allsederr[:,1]-allimerr[:,1]<0) & (allsederr[:,3]-allimerr[:,3]<0) & (allsederr[:,4]-allimerr[:,4]<0) & (allsederr[:,5]-allimerr[:,5]<0))[0]
print(len(better_img), len(better_img)/len(np.where(theta[train_index:][:,4]>3)[0]))
plt.scatter(mvals[better_img], sfrvals[better_img], color='dodgerblue', alpha=1)
# plt.scatter(mvals[better_sed], sfrvals[better_sed], color='green', alpha=1)
plt.scatter(mvals, sfrvals, color='red', alpha=0.005)
plt.plot(mvals, (0.84-0.026*12.5)*mvals - (6.51-0.11*12.5), color='k', linestyle='--')

plt.figure()
plt.hist(allsederr[:,1][better_img]-allimerr[:,1][better_img])
# print(len(np.where(allsederr[:,1][better_img]-allimerr[:,1][better_img]>0)[0]))
print(np.nanmean(sersic_n[better_img]))
print(np.nanmean(sersic_n[better_sed]))

In [ ]:
np.nanmean(scatter_5_band_sed), np.nanmean(scatter_im), np.nanmean(scatter_true), np.nanstd(scatter_5_band_sed), np.nanstd(scatter_im), np.nanstd(scatter_true)

In [ ]:
len(mask_im), len(mask_sed), len(mask_true)

In [ ]:
mvals = theta[train_index:,0]
sfrvals = theta[train_index:,1] + theta[train_index:,0] - 9

In [ ]:
#Galaxies on SFMS
better_img = np.where((allsederr[:,1]-allimerr[:,1]>0) & (np.array(theta[:,1])[train_index:]<=-1.5)  & (np.array(theta[:,1])[train_index:]>=-2.5) & (sersic_n>=4)  & (sersic_n<6))[0]
better_sed = np.where((allsederr[:,1]-allimerr[:,1]<0) & (np.array(theta[:,1])[train_index:]<=-1.5) & (np.array(theta[:,1])[train_index:]>=-2.5) & (sersic_n>=4) & (sersic_n<6))[0]
print(len(better_img), len(better_img)/len((np.where((np.array(theta[:,1][train_index:])>=0) &(bt>=0.9)))[0]))
print(len(better_sed))
plt.scatter(mvals[better_img], sfrvals[better_img], color='dodgerblue', alpha=1)
# plt.scatter(mvals[better_sed], sfrvals[better_sed], color='green', alpha=1)
plt.scatter(mvals, sfrvals, color='red', alpha=0.005)
plt.plot(mvals, (0.84-0.026*12.5)*mvals - (6.51-0.11*12.5), color='k', linestyle='--')

plt.figure()
plt.hist(allsederr[:,1][better_img]-allimerr[:,1][better_img])
plt.figure()
plt.hist(sersic_n[better_img])
plt.figure()
plt.hist(sersic_n[better_sed])
# print(len(np.where(allsederr[:,1][better_img]-allimerr[:,1][better_img]>0)[0]))
print(np.nanmean(sersic_n[better_img]))
print(np.nanmean(sersic_n[better_sed]))

In [ ]:
posterior_ensemble_sed , summaries_sed = hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_5_bands_SED_train_set.hickle')
posterior_ensemble_im, summaries_im = hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/clean_trained_ILIs/trained_ILI_images_train_set.hickle')
gswlc_embeds =  hickle.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/ViT_embeds/gswlc_emebds_clean.hickle')
train_index = int(0.8*len(gswlc_data))

In [ ]:
len(better_img)

In [ ]:
print(np.nanmean(sersic_n[better_img]))
print(np.nanmean(sersic_n[better_sed]))

In [ ]:
plt.hist(pbulge)
print(np.nanmax(pbulge))

In [ ]:
mask_low_bulge =np.where((allimpred[:,0]>11) & (sfr_vals_im>=main_sequence_sfr_im-0.5)&((sfr_vals_im<=main_sequence_sfr_im+0.5)))[0]
plt.hist(pbulge[mask_low_bulge])

In [ ]:
#make a subplot to show all the images in a grid
# selected_sample = better_img
# selected_sample = np.arange(0, 100)
# selected_sample = np.where((allimpred[:,0]<9)&(pbulge>0.9))[0][:100]
selected_sample = np.where((mvals_im >8.5)&(mvals_im<9)&((sfr_vals_im>=main_sequence_sfr_im-0.4)))[0][:100]
plt.subplots(int(np.sqrt(len(selected_sample))),int(np.sqrt(len(selected_sample))),figsize=(20,20))
for i in range(len(selected_sample)):
    j = selected_sample[i] + train_index
    plt.subplot(int(np.sqrt(len(selected_sample))),int(np.sqrt(len(selected_sample))),i+1)
    # plt.title('Sersic n: '+str(sersic_n[j-train_index])[0:4])
    plt.title('Pbulge: '+str(pbulge[j-train_index])[0:4])
    plt.imshow(Image.open(io.BytesIO(gswlc_images[j])))
    plt.axis('off')
        # plt.show()

plt.show();

In [ ]:
clrs = plt.cm.Spectral(np.linspace(0,1,10))
ranges = [(8,12),(-4.5,3),(0,0.16),(0,13.7),(0,4),(6.5,9.5)]
labels = ['log M$_*$', 'log sSFR','redshift','t50','Av','Z']

all_covs_sed_sub_sample, all_samples_sed_sub_sample = [], []
all_covs_im_sub_sample, all_samples_im_sub_sample = [], []

# rand_index = np.random.choice(better_img, 5, replace=False)

for gid in (better_img):

    gid = gid + train_index #making sure we take data from the training
    # xobs_sed = eleven_bands_SED[gid,:]
    xobs_sed = five_bands_SED[gid,:]
    # xobs_sed = three_bands_SED[gid,:]
    # xobs_im = infer(Image.open(io.BytesIO(gswlc_images[gid])))
    xobs_im = gswlc_embeds[gid,:]
    x_truth = theta[gid,:]

    posterior_samples_sed = posterior_ensemble_sed.sample((1000,), torch.Tensor(xobs_sed).to(device), show_progress_bars = False)
    posterior_samples_sed = posterior_samples_sed.cpu().numpy()
    # covs_sed, _ = check_degeneracy(posterior_samples_sed, vb=True)

    posterior_samples_im = posterior_ensemble_im.sample((1000,), torch.Tensor(xobs_im).to(device), show_progress_bars = False)
    posterior_samples_im = posterior_samples_im.cpu().numpy()
    # covs_im, _ = check_degeneracy(posterior_samples_im, vb=True)

    # all_covs_sed.append(covs_sed)
    all_samples_sed.append(posterior_samples_sed)
    # all_covs_im.append(covs_im)
    all_samples_im.append(posterior_samples_im)

    print(x_truth)

    plt.imshow(Image.open(io.BytesIO(gswlc_images[gid])))
    plt.axis('off')

    fig = corner.corner(posterior_samples_sed, truths=x_truth,color='dodgerblue',range=ranges,labels=labels, plot_datapoints=False, fill_contours=True,levels=[1 - np.exp(-(1/1)**2/2),1 - np.exp(-(2/1)**2/2)])
    fig = corner.corner(posterior_samples_im, truths=x_truth, truth_color='k',color='firebrick',range=ranges,labels=labels,fig=fig, plot_datapoints=False, fill_contours=True,levels=[1 - np.exp(-(1/1)**2/2),1 - np.exp(-(2/1)**2/2)])
    fig.set_size_inches(12,12)
    plt.show()

In [ ]:
mask = np.where(np.array(allimpred)[:,1] > -99)[0]
plt.figure()
plt.scatter(np.array(allimpred)[:,0][mask],np.array(allimpred)[:,1][mask]+np.array(allimpred)[:,0][mask]-9, color='dodgerblue', alpha=0.5)
plt.plot(np.array(allimpred)[:,0][mask], (0.84-0.026*12.5)*np.array(allimpred)[:,0][mask] - (6.51-0.11*12.5), color='k', linestyle='--')

plt.figure()
plt.plot(np.array(allsedpred)[:,0][mask], (0.84-0.026*12.5)*np.array(allsedpred)[:,0][mask] - (6.51-0.11*12.5), color='k', linestyle='--')
plt.scatter(np.array(allsedpred)[:,0][mask],np.array(allsedpred)[:,1][mask]+np.array(allsedpred)[:,0][mask]-9, color='dodgerblue', alpha=0.5)

# UMAP of Images Embeddings

In [ ]:
import umap

# data = np.array(gswlc_embeds)
# # Create UMAP reducer
# reducer = umap.UMAP()

# # Fit and transform the data
# embedding = reducer.fit_transform(data)

# Visualize the embedding (if you want 2D)
plt.hexbin(embedding[:, 0], embedding[:, 1], C = gswlc_data_merged['pbulge_morph'], gridsize=40) # s=5 for smaller dots
plt.title("UMAP Embedding for Untrained ViT")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.colorbar(label='Prominent Bulge')
plt.show()

In [ ]:
plt.hexbin(embedding[:, 0], embedding[:, 1], C = gswlc_data['t_50'], gridsize=40) # s=5 for smaller dots
plt.title("UMAP Embedding for Untrained ViT")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.colorbar(label='t$_{50}$')
plt.show()

# Morphological Fitting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
from astropy.modeling.models import Sersic2D
from astropy.convolution import convolve, Gaussian2DKernel, Moffat2DKernel
from photutils.segmentation import detect_threshold, detect_sources
import time
import statmorph 

import warnings
warnings.filterwarnings("ignore")

fwhm_pix = 0.262

beta = 4
gamma = fwhm_pix / (2.0 * np.sqrt(2**(1.0/beta) - 1.0))
kernel = Moffat2DKernel(gamma=gamma, alpha=beta)
kernel.normalize()  # make sure kernel adds up to 1
psf = kernel.array  # we only need the numpy array

sersic_n = []
sersic_chi_sq = []
sersic_flag = []
r20 = []
r80=[]
full_morph=[]

# for gid in tqdm(range(train_index, len(gswlc_data))):
for gid in tqdm(range(train_index, train_index+1000)):
    image = np.array(Image.open(io.BytesIO(gswlc_images[gid])))[:,:,0]
    np.random.seed(3)
    gain = 3
    # image = np.random.poisson(image * gain) / gain
    threshold = detect_threshold(image, 4)
    npixels = 5  # minimum number of connected pixels
    convolved_image = convolve(image, psf)
    segmap = detect_sources(image, threshold, npixels)
# plt.imshow(image, origin='lower', cmap='gray')
# plt.figure()
# plt.imshow(segmap, origin='lower', cmap='gray')
    try:
        source_morphs = statmorph.source_morphology(
            image, segmap, gain=gain, psf=psf);

        morph = source_morphs[0]
        sersic_n.append(morph.sersic_n)
        sersic_chi_sq.append(morph.sersic_chi2_dof)
        sersic_flag.append(morph.flag_sersic)
        r20.append(morph.r20)
        r80.append(morph.r80)
        full_morph.append(morph)
    except:
        sersic_n.append(np.nan)
        sersic_chi_sq.append(np.nan)
        sersic_flag.append(9)
        r20.append(np.nan)
        r80.append(np.nan)
        full_morph.append(np.nan)
        # print('error in image', gid)

sersic_n = np.array(sersic_n)
sersic_chi_sq = np.array(sersic_chi_sq)
sersic_flag = np.array(sersic_flag)
r20 = np.array(r20)
r80 = np.array(r80)
# full_morph = np.array(full_morph)
# np.save('full_morph.npy', full_morph)

In [ ]:
# mask = np.where(sersic_n < 6)[0]
plt.hexbin(mvals,sfrvals, 
           C=gswlc_data_merged['ttype_morph'][train_index:], gridsize=50, cmap='viridis',)
plt.colorbar()

sersic_n = np.array(gswlc_data_merged['bulge_n_morph'][train_index:])
bt = np.array(gswlc_data_merged['bt_morph'][train_index:])
pbulge = np.array(gswlc_data_merged['pbulge_morph'][train_index:])
ttype= np.array(gswlc_data_merged['ttype_morph'][train_index:])

In [ ]:
from statmorph.utils.image_diagnostics import make_figure
fig = make_figure(full_morph[better_img][8])

In [ ]:
plt.hist(np.array(theta[:,2])[better_img])
plt.hist(np.array(theta[:,2])[better_sed])
print(np.mean(np.array(theta[:,2])[better_img]), np.mean(np.array(theta[:,2])[better_sed]))

In [ ]:
# clean_image = np.where(sersic_flag[better_img]<=1)[0]
# clean_sed = np.where(sersic_flag[better_sed]<=1)[0]
plt.hist(sersic_n[better_img])
plt.hist(sersic_n[better_sed])
print(np.nanmedian(sersic_n[better_img]), np.nanmedian(sersic_n[better_sed]))
print(len(sersic_n[better_img][~np.isnan(sersic_n[better_img])]), len(sersic_n[better_sed][~np.isnan(sersic_n[better_sed])]))
print(len(better_img), len(better_sed))

In [ ]:
len(sersic_n), len(np.unique(sersic_n))

In [ ]:
plt.hist(sersic_n)
print(np.nanmedian(sersic_n),np.nanstd(sersic_n))

In [ ]:
mask = np.where(sersic_n < 6)[0]
plt.hexbin(mvals[mask],sfrvals[mask], 
           C=sersic_n[mask], gridsize=50, cmap='viridis', vmin=0,vmax=6)
plt.colorbar()

In [ ]:
morph_data = pd.read_csv('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/matched_morph_data_v3.csv')

In [ ]:
morph_data['ttype_morph']

In [ ]:
if 'ID' not in gswlc_data.columns:
        print("\nError: 'ID' column not found in gswlc_data.")
elif 'ID' not in morph_data.columns:
    print("\nError: 'ID' column not found in morph_data.")
# Check if 'sersic_n_morph' exists in morph_data
elif 'bt_morph' not in morph_data.columns:
        print("\nError: 'bt_morph' column not found in morph_data.")
else:
    # Optional: Ensure 'ID' columns have the same data type if needed
    # gswlc_data['ID'] = gswlc_data['ID'].astype(int)
    # morph_data['ID'] = morph_data['ID'].astype(int)
    print("\nData loaded successfully. Proceeding with merge...")

    # --- Perform the left merge ---
    # We select only the 'ID' and 'sersic_n_morph' columns from morph_data
    gswlc_data_merged = pd.merge(gswlc_data, morph_data[['ID', 'pbulge_morph','bt_morph','half_light_radius_morph', 'bulge_n_morph', 'disk_n_morph', 'ttype_morph']], on='ID', how='left')

    # --- Display results ---
    print("\nMerge complete. Updated gswlc_data DataFrame head:")
    print(gswlc_data_merged.head())

    # Check for NaN values in the new column
    print("\nNumber of NaN values in 'bulge_n_morph' after merge:")
    print(gswlc_data_merged['bulge_n_morph'].isnull().sum())

    # You might want to assign the merged result back or use the new DataFrame:
    # gswlc_data = gswlc_data_merged


In [ ]:
plt.hist(gswlc_data_merged['ttype_morph'][train_index:])

tester_array = np.array(gswlc_data_merged['bulge_n_morph'][train_index:])
len(tester_array), len(tester_array[~np.isnan(tester_array)])